In [ ]:
"""
Sistema de clasificación de intenciones usando KNN para predicciones de inventario.
"""

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
import pickle
import json
from typing import Dict, Any, Tuple
from datetime import datetime
import re

class IntentClassifier:
    """
    Clasificador de intenciones usando KNN para determinar qué endpoint usar.
    """
    
    def __init__(self):
        self.vectorizer = TfidfVectorizer(
            max_features=100,
            ngram_range=(1, 3),
            stop_words=None  # Personalizar según necesites
        )
        self.knn = KNeighborsClassifier(
            n_neighbors=3,
            weights='distance',
            metric='cosine'
        )
        self.intent_labels = {
            0: "predict_out_of_stock",           # /predict/out-of-stock
            1: "predict_product_stock",          # /predict/product-stock
            2: "predict_date",                   # /predict/date
            3: "predict_product_out_of_stock"    # /predict/product-out-of-stock
        }
        self.is_trained = False
    
    def get_training_data(self):
        """
        Dataset de entrenamiento con ejemplos de consultas y sus intenciones.
        """
        training_examples = [
            # Intent 0: predict_out_of_stock (productos que se agotarán pronto)
            ("cual producto se agotará primero", 0),
            ("qué productos se quedarán sin stock", 0),
            ("cuál es el primer producto en agotarse", 0),
            ("productos que se van a acabar pronto", 0),
            ("qué se acaba primero", 0),
            ("productos con riesgo de agotamiento", 0),
            ("cuáles productos están en riesgo", 0),
            ("productos que se agotarán próximamente", 0),
            ("siguiente producto sin stock", 0),
            ("qué producto tiene más riesgo", 0),
            ("qué artículos están por agotarse", 0),
("dime qué se quedará sin stock pronto", 0),
("qué producto está más cerca de acabarse", 0),
("qué inventario caerá primero a cero", 0),
("productos casi sin unidades", 0),
("cuáles artículos se acabarán antes", 0),
("dame los productos que tienen menos stock futuro", 0),
("qué producto tiene menos tiempo antes de agotarse", 0),
("cuál se termina antes que los demás", 0),
("productos con pocas unidades restantes", 0),
("qué referencias están al borde del agotamiento", 0),
("lista los productos que bajarán a cero pronto", 0),
("qué productos están próximos a agotarse", 0),
("inventario crítico por agotarse", 0),
("qué artículo tiene riesgo inmediato", 0),
("cuáles productos tienen stock crítico", 0),
("saber qué productos están próximos a acabarse", 0),
("productos con alerta de agotamiento", 0),
("artículos con pocas existencias", 0),
("cuál es el inventario más bajo", 0),
("qué productos ya casi no quedan", 0),
("productos que se terminarán en poco tiempo", 0),
("lista de artículos próximos a acabarse", 0),
("cuáles stocks caerán a cero primero", 0),
("dime los productos con mayor riesgo", 0),
("qué se está consumiendo más rápido", 0),
("qué inventarios están casi terminados", 0),
("productos que pronto no estarán disponibles", 0),
("candidatos a agotarse esta semana", 0),
("qué artículo se agotará antes del resto", 0),
("cuáles referencias presentan déficit cercano", 0),
("cuáles unidades están a punto de acabarse", 0),
("productos que requieren reposición urgente", 0),
("qué producto tiene menos unidades disponibles", 0),
("items que no alcanzarán para mucho tiempo", 0),
("artículos que deben reponerse de inmediato", 0),
("mercancía que va a escasear pronto", 0),
("qué se terminará pronto según tendencias", 0),
("inventario que bajará a cero pronto", 0),
("qué referencia desaparecerá primero del stock", 0),
("productos que van a agotarse por demanda", 0),
("qué tiene menor previsión de stock", 0),
("productos que deben reabastecerse ya", 0),
("artículos que muestran agotamiento próximo", 0),
("qué stocks bajan más rápido", 0),
("cuáles referencias están casi agotadas", 0),
("cuáles productos no alcanzarán para mañana", 0),
("artículos en peligro de quedarse sin stock", 0),
("cerca de agotarse: qué productos", 0),
("dame los productos que se están quedando sin stock", 0),
("qué artículos están en riesgo alto", 0),
("inventario con riesgo crítico", 0),
("qué unidades quedan muy pocas", 0),
("productos que podrían agotarse hoy", 0),
("qué se va a agotar en las próximas horas", 0),
("ítems que se van a acabar en breve", 0),
("top productos por agotarse", 0),
("cuáles productos tienen el stock más bajo hoy", 0),
("qué productos quedarán sin stock muy pronto", 0),
("qué inventario tiene tendencia a cero", 0),
("productos que desaparecerán del inventario", 0),
("artículos por agotar en los próximos días", 0),
("productos con disponibilidad mínima", 0),
("qué artículo está en nivel crítico", 0),
("qué productos tienen umbral crítico", 0),
("cuál es el siguiente producto en agotarse", 0),
("productos que necesitan reabastecimiento inmediato", 0),
("cuáles stocks están en límites bajos", 0),
("cuáles productos se acabarán por alta demanda", 0),
("ítems con menor previsión futura", 0),
("qué mercancía está casi acabada", 0),
("productos que presentan próximo agotamiento", 0),
("productos sin stock pronto", 0),
("cuáles artículos van a tocar cero unidades", 0),
("qué productos ya casi se terminan", 0),
("qué inventarios se reducen demasiado rápido", 0),
("productos a punto de agotarse completamente", 0),
("qué se queda sin existencias antes", 0),
("cuáles items deben reponerse urgente", 0),
("productos con fecha de agotamiento cercana", 0),
("dime los artículos próximos a agotarse", 0),
("qué productos llegan al límite pronto", 0),
("inventario a punto de acabarse", 0),
("cuál producto está casi agotado", 0),
("qué artículos se irán a cero antes", 0),
("cuál es el stock con menor duración futura", 0),
("qué inventario está más comprometido", 0),
("productos que quedarán sin unidades disponibles", 0),
("qué se agotará según las predicciones", 0),
("cuáles productos tienen caída más rápida", 0),
("productos que ya no tendrán stock", 0),
("artículos bajo umbral mínimo", 0),
("qué referenceas presentan agotamiento próximo", 0),
("productos al borde del desabastecimiento", 0),
("qué unidades son insuficientes", 0),
("artículos con inventario casi nulo", 0),
("cuáles se terminan hoy", 0),
("qué existencias se acabarán en corto plazo", 0),
("productos en riesgo inmediato de agotarse", 0),
("qué ítems quedarán en cero primero", 0),
("qué productos tienen urgencia de reposición", 0),
("productos que estarán agotados mañana", 0),

            
            # Intent 1: predict_product_stock (stock de producto específico en fecha)
            ("cuánto stock tendrá producto X el 15 de mayo", 1),
            ("stock de producto Y para el 2025-06-10", 1),
            ("cuántas unidades de producto Z habrá el próximo mes", 1),
            ("predicción de stock de producto A en fecha específica", 1),
            ("stock producto B para el día 20", 1),
            ("cantidad de producto C el 15 de marzo", 1),
            ("unidades de producto D en una fecha", 1),
            ("stock futuro de producto E", 1),
            ("cuánto quedará de producto F el día X", 1),
            ("predicción producto G fecha Y", 1),
            ("cuánto stock tendrá el producto X en fecha Y", 1),
("cantidad prevista para producto X el día 15", 1),
("qué unidades habrá del producto A el próximo viernes", 1),
("proyección de stock de producto Z para junio", 1),
("cuántas existencias tendrá artículo B en fecha futura", 1),
("stock esperado de producto C el día 10", 1),
("cuánto inventario habrá de item X el mes siguiente", 1),
("cuántas unidades quedarán de ese producto el 2025-01-01", 1),
("predicción del stock para producto X", 1),
("unidades futuras del artículo Y", 1),
("proyección exacta del stock del producto A", 1),
("cuánto queda del producto Z en la fecha dada", 1),
("previsión de existencias del producto C", 1),
("qué cantidad habrá del artículo X en tal fecha", 1),
("stock disponible para producto Y en fecha señalada", 1),
("cuánto inventario tendrá item F el día indicado", 1),
("cuántas unidades habrá del producto solicitado", 1),
("proyección futura para referencia X", 1),
("cuál será el stock del producto en X fecha", 1),
("unidades futuras estimadas del producto C", 1),
("inventario proyectado de ese artículo", 1),
("cantidad futura del producto A", 1),
("predicción del nivel de stock del producto B", 1),
("unidades estimadas de referencia X para fecha Y", 1),
("cuánto stock habrá el próximo mes del producto Z", 1),
("estimación de existencias futuras", 1),
("cuántos quedan en tal fecha", 1),
("unidades del producto X en el día Y", 1),
("inventario estimado para producto A", 1),
("cantidad disponible en el futuro para el producto B", 1),
("proyección del inventario para una fecha concreta", 1),
("cuál será la cantidad disponible del artículo X", 1),
("predicción de cuántas unidades quedarán", 1),
("estimación futura de stock", 1),
("cuánto quedará del producto en fecha futura", 1),
("nivel de stock futuro del artículo", 1),
("existencias del producto A el próximo 10 de mayo", 1),
("unidades en inventario para producto Z en una fecha", 1),
("cantidad proyectada para producto X", 1),
("cuántas unidades habrá el día 20 del producto A", 1),
("predicción de unidades restantes", 1),
("stock del producto C para futuro cercano", 1),
("cuántas unidades quedarán en fecha dada", 1),
("cuánto stock habrá para producto Y en el día pedido", 1),
("inventario futuro de artículo Z", 1),
("stock proyectado para referencia X", 1),
("cantidad estimada en días futuros para producto B", 1),
("proyección numérica del stock", 1),
("cuántas unidades tendrá disponible ese artículo", 1),
("cuántos productos quedarán en la fecha consultada", 1),
("predictivo del stock para un producto específico", 1),
("inventario estimado para artículo X", 1),
("stock pronosticado del producto A", 1),
("cuánto quedará disponible del producto Z", 1),
("cantidad futura del artículo requerido", 1),
("valor estimado del stock para ese producto", 1),
("cuántas unidades estarán disponibles en fecha X", 1),
("nivel estimado para el producto A en el día 12", 1),
("dame la cantidad futura del producto X", 1),
("cuánto stock quedará del producto en esa fecha", 1),
("cuántas unidades tendrá el inventario del artículo Y", 1),
("previsión de unidades del producto C", 1),
("cantidad futura estimada producto D", 1),
("inventario del producto en el día solicitado", 1),
("cuántas existencias tendrá producto X el próximo lunes", 1),
("existencias previstas para producto Z", 1),
("unidades futuras disponibles para referencia A", 1),
("cuánto inventario habrá más adelante", 1),
("ver stock futuro del producto X", 1),
("cantidad que quedará para ese artículo", 1),
("unidades pronosticadas del producto A", 1),
("cuál será el inventario del producto Z el día 5", 1),
("cuántos artículos habrá en la fecha establecida", 1),
("cantidad esperada en el futuro", 1),
("proyección del nivel de existencias", 1),
("stock futuro para producto concreto", 1),
("existencias en el tiempo para ese producto", 1),
("cuántos quedarán de este producto el día 15", 1),
("cantidad proyectada del producto consultado", 1),
("cuánto stock tendrá el producto B en agosto", 1),
("producto X: unidades futuras", 1),
("inventario proyectado del producto Y", 1),
("predicción de disponibilidad futura del producto", 1),
("cantidad pronosticada en fecha determinada", 1),
("unidades futuras calculadas para ese producto", 1),
("cuántas unidades quedarán mañana", 1),
("proyección exacta del inventario del artículo", 1),
("cantidad futura que tendrá el producto Y", 1),
("cuánto stock habrá en fecha específica", 1),
("cuántas unidades estarán en inventario ese día", 1),
("dime la cantidad futura del artículo solicitado", 1),
("cuántas existencias tendrá el producto el 25", 1),
("previsión para stock de referencia X", 1),
("nivel de stock futuro para el producto indicado", 1),
("proyección de cuántas unidades habrá", 1),
("cantidad de stock futura para artículo A", 1),

            
            # Intent 2: predict_date (todo el inventario en una fecha)
            ("cómo estará el inventario el 20 de junio", 2),
            ("estado del inventario para el 2025-07-15", 2),
            ("todos los productos en fecha X", 2),
            ("inventario completo el próximo mes", 2),
            ("qué productos y cantidades habrá el día 30", 2),
            ("predicción de todo el stock para una fecha", 2),
            ("estado general del inventario en fecha", 2),
            ("todos los stocks en el futuro", 2),
            ("inventario total el día X", 2),
            ("predicción completa para fecha Y", 2),
            ("inventario total proyectado para fecha X", 2),
("dime todo el inventario disponible el próximo lunes", 2),
("qué habrá en todo el almacén en esa fecha", 2),
("estado completo de existencias para futuro", 2),
("inventario general para día especificado", 2),
("stocks futuros de todos los productos", 2),
("cómo estará todo el inventario en fecha determinada", 2),
("cantidad total por producto en el futuro", 2),
("listado de inventario futuro", 2),
("predicción del inventario completo", 2),
("qué habrá en almacén para el mes siguiente", 2),
("panorama del inventario en tal fecha", 2),
("inventario proyectado del almacén", 2),
("qué cantidades totales habrá en esa fecha", 2),
("dame todos los productos y sus unidades futuras", 2),
("inventario completo del día X", 2),
("existencias totales en el futuro", 2),
("stock global para fecha específica", 2),
("estado del sistema de inventario futuro", 2),
("total de existencias para un día", 2),
("cómo lucirá todo el inventario para mañana", 2),
("predicción del inventario general", 2),
("cuál será el inventario total en X fecha", 2),
("lista completa de productos y existencias futuras", 2),
("cuántas unidades habrá de cada artículo", 2),
("inventario completo estimado", 2),
("existencias proyectadas del almacén entero", 2),
("visión completa del stock en fecha dada", 2),
("disponibilidad total proyectada", 2),
("resumen de inventario para día X", 2),
("dime el stock futuro de todos los artículos", 2),
("inventario completo según predicción", 2),
("qué unidades habrá de cada producto", 2),
("informe de inventario futuro", 2),
("total de stocks proyectados", 2),
("estado global del inventario", 2),
("inventario total estimado para una fecha", 2),
("qué habrá disponible en todo el inventario", 2),
("inventario completo del almacén para fecha Y", 2),
("stocks futuros de todos los ítems", 2),
("inventario esperado en la fecha consultada", 2),
("predicción del sistema de inventario completo", 2),
("dame todo el stock disponible en esa fecha", 2),
("cómo estará el almacén completo", 2),
("existencias totales para día futuro", 2),
("predicción total de existencias del almacén", 2),
("qué inventario habrá en la fecha elegida", 2),
("todos los productos y sus cantidades en fecha X", 2),
("cuántas unidades habrá por cada referencia", 2),
("inventario proyectado del sistema completo", 2),
("vista general del inventario del futuro", 2),
("total de productos disponibles en fecha dada", 2),
("inventario futuro de la empresa", 2),
("existencias del almacén en fecha futura", 2),
("listado general de inventario proyectado", 2),
("resumen futuro de existencias", 2),
("predicción del stock total", 2),
("estado general del almacén", 2),
("qué tan lleno estará el inventario en esa fecha", 2),
("inventario global del sistema en fecha específica", 2),
("cuánto habrá en total de cada producto", 2),
("listado completo del inventario en el futuro", 2),
("estado futuro del inventario de todos los productos", 2),
("inventario completo para mañana", 2),
("inventario completo disponible el próximo viernes", 2),
("existencias completas del almacén", 2),
("cuál será el panorama del inventario", 2),
("stock total del inventario", 2),
("qué existencias habrá de forma global", 2),
("cómo estará distribuido el inventario", 2),
("inventario global estimado", 2),
("predicción total del almacén", 2),
("todos los items y sus unidades futuras", 2),
("inventario global futuro para una fecha fija", 2),
("resumen detallado del inventario futuro", 2),
("estado total de existencias de todos los artículos", 2),
("informe completo del inventario futuro", 2),
("total de existencias en el almacén entero", 2),
("cuánto habrá disponible de todo el inventario", 2),
("visión proyectada del inventario total", 2),
("predicción completa del stock del almacén", 2),
("estado futuro de stocks completos", 2),
("inventario del sistema en fecha futura", 2),
("qué cantidad total habrá disponible", 2),
("inventario reunido de todos los productos", 2),
("existencias totales proyectadas para el día X", 2),
("inventario en su totalidad para la fecha", 2),
("qué existencias globales habrá en fecha futura", 2),
("lista total de productos y stock futuro", 2),
("nivel global del inventario para fecha Y", 2),
("proyección total de unidades del inventario", 2),

            
            # Intent 3: predict_product_out_of_stock (cuándo se agota producto específico)
            ("cuándo se agotará producto X", 3),
            ("fecha de agotamiento de producto Y", 3),
            ("cuándo se acaba producto Z", 3),
            ("en qué fecha se quedará sin stock producto A", 3),
            ("cuándo se termina producto B", 3),
            ("fecha límite de producto C", 3),
            ("cuándo producto D llegará a cero", 3),
            ("predicción de agotamiento producto E", 3),
            ("cuándo no habrá más producto F", 3),
            ("fecha sin stock producto G", 3),
            ("en qué día se quedará sin stock este producto", 3),
("fecha exacta en que el producto llegará a cero", 3),
("cuándo terminará el inventario del producto A", 3),
("dime cuándo se agotará el artículo X", 3),
("predicción de fecha de agotamiento", 3),
("cuándo se quedarán sin existencias del producto B", 3),
("en qué momento el producto Z llegará a cero unidades", 3),
("cuál es la fecha en que se acaba el producto A", 3),
("cuándo no quedará stock del artículo Y", 3),
("línea de tiempo para agotarse el producto X", 3),
("cuándo se termina ese producto del inventario", 3),
("proyección de cuándo se queda sin stock", 3),
("cuándo será el agotamiento del producto consultado", 3),
("en qué fecha toca cero unidades", 3),
("cuándo se vacía el inventario del producto", 3),
("día estimado de agotamiento", 3),
("cuándo se acaba definitivamente el producto Z", 3),
("fecha donde ya no habrá unidades del artículo", 3),
("cuándo dejará de estar disponible el producto", 3),
("predicción del día que se agota", 3),
("cuándo quedará en cero el stock", 3),
("momento exacto donde se acaba el producto", 3),
("proyección de ruptura de stock", 3),
("cuándo no habrá más existencias del producto", 3),
("cuál será la fecha del agotamiento total", 3),
("día en que el artículo quedará sin unidades", 3),
("cuándo el stock del producto llegará a cero", 3),
("fecha estimada en que se termina este artículo", 3),
("cuándo se quedará vacío el inventario de ese producto", 3),
("cuándo se agotará completamente", 3),
("momento futuro donde se acaba ese producto", 3),
("cuándo se deja de tener stock del producto", 3),
("fecha prevista de finalización del inventario", 3),
("cuándo no quedará ninguna unidad disponible", 3),
("a qué fecha llegará a cero el producto B", 3),
("dime el día que se agota la referencia X", 3),
("cuándo dejará de existir stock", 3),
("cuándo toca cero el inventario del artículo", 3),
("predicción del fin de stock", 3),
("cuándo se acaba el artículo requerido", 3),
("cuándo no habrá más en inventario", 3),
("cuándo el sistema marcará cero unidades", 3),
("fecha exacta de agotamiento proyectado", 3),
("cuándo llegará al mínimo absoluto", 3),
("cuándo desaparece del stock el producto X", 3),
("cuánto falta para que se agote ese artículo", 3),
("día estimado sin existencias", 3),
("en qué fecha alcanza cero unidades el producto", 3),
("cuándo será el desabastecimiento del producto", 3),
("fecha proyectada donde se acaba", 3),
("cuándo no habrá más unidades del artículo Z", 3),
("cuándo el producto se queda sin inventario", 3),
("dime la fecha de agotamiento del producto", 3),
("cuándo el inventario quedará vacío", 3),
("cuando desaparecerá el stock del producto", 3),
("día previsto donde no habrá existencias", 3),
("proyección de fecha sin stock", 3),
("cuándo se consume completamente el inventario", 3),
("cuándo termina el inventario del artículo", 3),
("fecha exacta donde el stock baja a cero", 3),
("cuándo se agota ese ítem", 3),
("cuándo cae a cero el producto A", 3),
("momento donde no habrá más producto", 3),
("cuándo se vaciará el inventario", 3),
("día en que se agotan las existencias", 3),
("cuándo finalizan unidades del producto", 3),
("fecha para la que ya no habrá stock disponible", 3),
("cuándo se termina el suministro de ese producto", 3),
("cuándo el artículo dejará de existir en inventario", 3),
("fecha en que ya no habrá producto Z", 3),
("cuándo se quedará el almacén sin ese producto", 3),
("cuándo se proyecta el agotamiento de unidades", 3),
("qué día el producto llegará a cero", 3),
("cuándo ya no se podrá vender ese producto", 3),
("fechas posibles en que se acaba el producto", 3),
("cuándo se extinguirá el stock", 3),
("cuándo estará agotado ese artículo", 3),
("fecha estimada sin unidades restantes", 3),
("momento en que se completa el agotamiento", 3),
("cuándo ocurre el fin del inventario", 3),
("fecha final donde no habrá más unidades", 3),
("a qué fecha se prevé que se acabe", 3),
("cuándo quedará sin existencias ese producto", 3),
("día proyectado para el agotamiento total", 3),
("cuándo no hay más stock disponible", 3),
("cuándo exactamente se termina ese artículo", 3),
("en qué fecha ya no habrá inventario", 3),
("cuándo se producirá la ruptura de stock", 3),
("cuándo será el último día con existencias", 3),
("cuándo desaparecerán las unidades en inventario", 3),
("momento proyectado del agotamiento", 3),
("cuándo se agota según las predicciones", 3),
("cuándo el stock del producto se vuelve nulo", 3),
("cuándo se quedará vacío el sistema", 3)

        ]
        
        texts = [example[0] for example in training_examples]
        labels = [example[1] for example in training_examples]
        
        return texts, labels
    
    def train(self):
        """
        Entrena el modelo KNN con los datos de ejemplo.
        """
        texts, labels = self.get_training_data()
        
        # Vectorizar textos
        X = self.vectorizer.fit_transform(texts)
        
        # Entrenar KNN
        self.knn.fit(X, labels)
        self.is_trained = True
        
        print("✓ Modelo KNN entrenado exitosamente")
        print(f"  - Muestras de entrenamiento: {len(texts)}")
        print(f"  - Características: {X.shape[1]}")
    
    def predict_intent(self, query: str) -> Tuple[str, float, int]:
        """
        Predice la intención del usuario a partir de su consulta.
        
        Args:
            query: Consulta del usuario en lenguaje natural
            
        Returns:
            Tuple con (nombre_endpoint, confianza, label_numerico)
        """
        if not self.is_trained:
            raise Exception("El modelo no ha sido entrenado. Llama a train() primero.")
        
        # Preprocesar consulta
        query_clean = query.lower().strip()
        
        # Vectorizar
        X = self.vectorizer.transform([query_clean])
        
        # Predecir
        prediction = self.knn.predict(X)[0]
        
        # Calcular confianza (distancia promedio a vecinos más cercanos)
        distances, _ = self.knn.kneighbors(X)
        confidence = 1 - (distances.mean() / 2)  # Normalizar a 0-1
        
        endpoint = self.intent_labels[prediction]
        
        return endpoint, confidence, prediction
    
    def extract_parameters(self, query: str, intent: str) -> Dict[str, Any]:
        """
        Extrae parámetros de la consulta según la intención detectada.
        
        Args:
            query: Consulta del usuario
            intent: Intención detectada
            
        Returns:
            Dict con los parámetros extraídos
        """
        query_lower = query.lower()
        params = {}
        
        # Extraer nombre de producto
        product_patterns = [
            r'producto\s+([a-z0-9_áéíóúñ]+)',
            r'del?\s+([a-z0-9_áéíóúñ]+)',
            r'de\s+([a-z0-9_áéíóúñ]+)',
        ]
        
        for pattern in product_patterns:
            match = re.search(pattern, query_lower)
            if match:
                params['product_name'] = match.group(1).strip()
                break
        
        # Extraer fecha
        date_patterns = [
            r'(\d{4}-\d{2}-\d{2})',  # 2025-05-15
            r'(\d{1,2})\s+de\s+(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre)',
            r'el\s+día\s+(\d{1,2})',
        ]
        
        for pattern in date_patterns:
            match = re.search(pattern, query_lower)
            if match:
                try:
                    # Intenta parsear la fecha
                    if '-' in match.group(0):
                        params['prediction_date'] = match.group(0)
                    else:
                        # Para otros formatos, usar fecha actual + días
                        params['prediction_date'] = match.group(0)
                    break
                except:
                    pass
        
        return params
    
    def save_model(self, path: str = "intent_classifier.pkl"):
        """
        Guarda el modelo entrenado en disco.
        """
        with open(path, 'wb') as f:
            pickle.dump({
                'vectorizer': self.vectorizer,
                'knn': self.knn,
                'intent_labels': self.intent_labels,
                'is_trained': self.is_trained
            }, f)
        print(f"✓ Modelo guardado en {path}")
    
    def load_model(self, path: str = "intent_classifier.pkl"):
        """
        Carga un modelo previamente entrenado.
        """
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.vectorizer = data['vectorizer']
            self.knn = data['knn']
            self.intent_labels = data['intent_labels']
            self.is_trained = data['is_trained']
        print(f"✓ Modelo cargado desde {path}")


# Ejemplo de uso
if __name__ == "__main__":
    # Crear y entrenar el clasificador
    classifier = IntentClassifier()
    classifier.train()
    
    # Probar con diferentes consultas
    test_queries = [
        "cual producto se va a agotar más pronto?",
        "cuánto stock tendrá producto A el 15 de mayo",
        "cómo estará el inventario el 20 de junio",
        "cuándo se agotará producto B",
    ]
    
    print("\n" + "="*60)
    print("PRUEBAS DE CLASIFICACIÓN")
    print("="*60)
    
    for query in test_queries:
        endpoint, confidence, label = classifier.predict_intent(query)
        params = classifier.extract_parameters(query, endpoint)
        
        print(f"\nConsulta: '{query}'")
        print(f"  → Endpoint: {endpoint}")
        print(f"  → Confianza: {confidence:.2%}")
        print(f"  → Parámetros: {params}")
    
    # Guardar el modelo
    classifier.save_model()